# NormLayer + LangGraph Demo

This notebook demonstrates how to use `LangGraphAdapter` to enforce behavioral policies
on messages produced by a LangGraph-style compiled graph.

**No real LangGraph or LLM calls are needed** — we use mock objects to simulate the graph.

In [ ]:
from normlayer import PolicyEngine
from normlayer.policies.loop_detection import LoopDetection
from normlayer.policies.role_respect import RoleRespect
from normlayer.adapters.langgraph_adapter import LangGraphAdapter

## 1. Define a mock compiled graph

In a real scenario, this would be a compiled `StateGraph`. Here we simulate one
that produces messages in a `"messages"` key.

In [ ]:
class MockMessage:
    """Simulates a LangGraph BaseMessage."""
    def __init__(self, content, type="ai", name=None):
        self.content = content
        self.type = type
        self.name = name
        self.additional_kwargs = {}

class MockGraph:
    """Simulates a compiled LangGraph StateGraph."""
    def __init__(self, output_messages):
        self._output = output_messages

    def invoke(self, state, **kwargs):
        result = dict(state)
        result["messages"] = state.get("messages", []) + self._output
        return result

## 2. Set up the engine and adapter

In [ ]:
engine = PolicyEngine(
    policies=[
        LoopDetection(max_repetitions=2, similarity_threshold=0.9, handler="warn"),
        RoleRespect(
            role_definitions={"planner": ["plan", "assign", "schedule"]},
            strict=True,
            handler="block",
        ),
    ]
)

adapter = LangGraphAdapter(engine)
print("Engine policies:", [p.name for p in engine.policies])

## 3. Clean run — no violations

In [ ]:
graph = MockGraph([
    MockMessage("I will plan the next sprint.", name="planner"),
])
wrapped = adapter.wrap(graph)

result = wrapped.invoke({"messages": []})
print(f"Messages in result: {len(result['messages'])}")
print(f"Violations so far: {len(engine.violations)}")

## 4. Trigger a RoleRespect violation

The planner agent tries to "execute" and "deploy" — outside its defined scope.

In [ ]:
from normlayer.engine import EnforcementError

bad_graph = MockGraph([
    MockMessage("I will execute the deployment now.", name="planner"),
])
wrapped_bad = adapter.wrap(bad_graph)

try:
    wrapped_bad.invoke({"messages": []})
except EnforcementError as e:
    print(f"Blocked! {e}")

print(f"\nAll violations ({len(engine.violations)}):")
for v in engine.violations:
    print(f"  - {v.policy_violated}: {v.message_snippet[:60]}")

## Summary

The `LangGraphAdapter` wraps any compiled graph's `invoke()` / `ainvoke()` and checks
new messages against the policy stack — no changes to your graph code required.